# 07. Final model

Final LightGBM on the final features, an alert-budget analysis (review the k riskiest transactions)
and saving the model.

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from src.features import FINAL_FEATURES
from src.split import make_masks
from src.models import make_lgbm, prepare_lgbm_frame
from src.evaluation import pick_threshold, metrics_at_threshold, precision_at_k
from src.score import score_transactions

In [ ]:
model_df = pd.read_parquet("../data/processed/model_df.parquet")
train_mask, val_mask, test_mask = make_masks(model_df)

y_train = model_df.loc[train_mask, "is_fraud"]
y_val = model_df.loc[val_mask, "is_fraud"]
y_test = model_df.loc[test_mask, "is_fraud"]

X_train = prepare_lgbm_frame(model_df.loc[train_mask], FINAL_FEATURES)
X_val = prepare_lgbm_frame(model_df.loc[val_mask], FINAL_FEATURES)
X_test = prepare_lgbm_frame(model_df.loc[test_mask], FINAL_FEATURES)

final_model = make_lgbm()
final_model.fit(X_train, y_train)

p_val = final_model.predict_proba(X_val)[:, 1]
p_test = final_model.predict_proba(X_test)[:, 1]

print("PR-AUC validation:", round(average_precision_score(y_val, p_val), 3))
print("PR-AUC test:", round(average_precision_score(y_test, p_test), 3))

## Threshold chosen on validation

In [ ]:
threshold = pick_threshold(p_val, y_val)
print("Threshold:", threshold)

pd.DataFrame([
    {"part": "validation", **metrics_at_threshold(y_val, p_val, threshold)},
    {"part": "test", **metrics_at_threshold(y_test, p_test, threshold)},
])[["part", "precision", "recall", "f1", "pr_auc", "TP", "FN", "FP"]].round(3)

## Alert budget

A fixed threshold hides what an analyst team really faces: a limited number of cases they can review.
Rank the test transactions by risk and look at the top k.

In [ ]:
budget = precision_at_k(y_test, p_test, ks=[500, 1000, 2000, 3000, 4000, 5000, 8000])
budget

In [ ]:
curve = precision_at_k(y_test, p_test, ks=list(range(250, 8001, 250)))

plt.plot(curve["k"], curve["precision@k"], label="precision@k")
plt.plot(curve["k"], curve["recall@k"], label="recall@k")
plt.xlabel("Reviewed transactions (top k by risk)")
plt.ylabel("Share")
plt.legend()

Path("../report").mkdir(exist_ok=True)
plt.savefig("../report/precision_recall_at_k.png", dpi=150, bbox_inches="tight")
plt.show()

## Save the model

In [ ]:
Path("../models").mkdir(exist_ok=True)
joblib.dump({"model": final_model, "columns": FINAL_FEATURES}, "../models/lgbm_fraud.joblib")

## Check: the scoring script gives the same numbers

`score_transactions` rebuilds the features from raw columns, the same way a new transaction would be scored.

In [ ]:
bundle = joblib.load("../models/lgbm_fraud.joblib")

sample = model_df.loc[test_mask].head(1000)
scores = score_transactions(bundle, sample)

print("Scoring script matches predict_proba:", np.allclose(scores.values, p_test[:1000]))